In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("open_clip") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch>=2.24,<3"])

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image as DisplayImage, display
cwd = Path.cwd().resolve()
candidates = [cwd / "inductive-bias-representations", cwd, cwd.parent]
candidates += list(cwd.glob("*/inductive-bias-representations"))
TASK_DIR = next((p for p in candidates if p.exists() and p.name == "inductive-bias-representations"), cwd / "inductive-bias-representations")
if str(TASK_DIR) not in sys.path:
    sys.path.insert(0, str(TASK_DIR))
from configs.task1_config import Config
from scripts.run_task1 import Task1Pipeline
CFG = Config()
pipeline = Task1Pipeline(CFG, TASK_DIR)
print({"task_dir": str(TASK_DIR), "device": str(pipeline.device), "run_tag": pipeline.run_tag})
assert pipeline.device.type == "cuda", "On Kaggle select Settings -> Accelerator -> GPU."

In [ ]:
subset_manifest = pipeline.prepare_data()
display(subset_manifest.groupby("class_name").size().rename("selected_count").to_frame().T)
display(subset_manifest.head())

In [ ]:
training_history = pipeline.train_heads()
display(training_history.groupby("model").tail(1))

In [ ]:
MANUAL_REJECTIONS = {}
candidates, accepted, qc_counts, qc_sheets = pipeline.generate_cue_conflicts(MANUAL_REJECTIONS)
display(qc_counts)
for sheet in qc_sheets:
    display(DisplayImage(filename=str(sheet), width=950))

In [ ]:
VISUAL_QC_REVIEW_COMPLETE = False
assert VISUAL_QC_REVIEW_COMPLETE, (
    "Paused for required visual QC. Record rejected IDs above, rerun generation if needed, "
    "then set VISUAL_QC_REVIEW_COMPLETE=True and run this cell again."
)

In [ ]:
feature_cache, conflict_features = pipeline.extract_evaluation_features()
print("Cached conditions:", sorted(feature_cache["resnet50"]))
print("Accepted cue conflicts:", len(pipeline.accepted))

In [ ]:
results = pipeline.evaluate()

In [ ]:
display(results["standard"].round(4))
display(DisplayImage(filename=str(pipeline.results_dir / "clean_color_patch_comparison.png"), width=1000))

In [ ]:
display(results["cue_conflicts"].round(2))
display(DisplayImage(filename=str(pipeline.results_dir / "cue_conflict_shape_bias_and_coverage.png"), width=900))
display(DisplayImage(filename=str(pipeline.results_dir / "cue_conflict_informative_examples.png"), width=1100))

In [ ]:
display(results["translation"].round(4))
display(DisplayImage(filename=str(pipeline.results_dir / "translation_curves.png"), width=1000))

In [ ]:
display(results["stability"].round(4))
display(results["alignment"].round(4))
display(DisplayImage(filename=str(pipeline.results_dir / "representation_cosine_stability.png"), width=900))
display(DisplayImage(filename=str(pipeline.results_dir / "prediction_representation_alignment.png"), width=850))

In [ ]:
for backbone in ["resnet50", "vit_b16", "clip_vit_b32"]:
    display(DisplayImage(filename=str(pipeline.results_dir / f"tsne_clean_vs_transformed_{backbone}.png"), width=1050))

In [ ]:
audit = pd.read_json(pipeline.results_dir / "completion_audit.json", typ="series")
display(audit.to_frame("value"))
print("Result files:")
for path in sorted(pipeline.results_dir.iterdir()):
    if path.is_file() and path.name != ".gitkeep":
        print(" -", path.name)